In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib import patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from pathlib import Path
import pandas as pd

In [ ]:
os.getcwd()
save_folder = 'output'

In [30]:
#  figure 2
"""
Figure 2: Route Distance Constraints and Return-Leg Battery Sizing
(Section 2.2 Route Distance Constraints)

Journal-refined styling — reproduces Figure_2_Route_Asymmetry.svg exactly.
Muted palette, thin lines, subtle gray spines/grid.
"""

# ── Exact palette extracted from reference SVG ──────────────────
BLUE      = "#3a5f7d"   # outbound legs, bars, arrowheads, legend solid
MAROON    = "#8b3a3a"   # return leg, return bar, hub markers, legend dashed
PORT_GRAY = "#2f2f2f"   # non-hub port markers
TEXT      = "#1a1a1a"   # titles, subtitles, labels, numbers, legend, axis labels
AXIS_GRAY = "#666666"   # tick marks + tick labels
GRID_GRAY = "#d8d8d8"   # grid lines


def _style_axes(ax):
    """Apply shared journal styling: gray left+bottom spines, gray ticks, faint grid."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color(AXIS_GRAY)
        ax.spines[side].set_linewidth(0.6)
    ax.tick_params(colors=AXIS_GRAY, width=0.6, labelcolor=AXIS_GRAY, labelsize=8)
    ax.grid(True, axis="y", color=GRID_GRAY, alpha=0.6, linewidth=0.4, linestyle="-")
    ax.set_axisbelow(True)


def figure_2_route_asymmetry():
    fig = plt.figure(figsize=(7.19, 5.28))

    ax_a_map = plt.subplot2grid((2, 2), (0, 0))
    ax_b_map = plt.subplot2grid((2, 2), (0, 1))
    ax_a_bar = plt.subplot2grid((2, 2), (1, 0))
    ax_b_bar = plt.subplot2grid((2, 2), (1, 1))

    # ── CASE A: T-9 route (asymmetric) ──────────────────────────
    t9_ports = [
        ("Surabaya",  112.73, -7.20),
        ("Makassar",  119.42, -5.13),
        ("Morotai",   128.48, 2.32),
        ("Galela",    127.98, 1.83),
        ("Maba",      128.22, 0.90),
        ("Weda",      127.90, 0.37),
    ]
    t9_leg_distances = [437, 870, 95, 80, 28, 1320]
    t9_leg_labels = ["SBY→MKS", "MKS→MRT", "MRT→GAL", "GAL→MAB", "MAB→WED", "WED→SBY"]

    # Per-port label offsets (points, matplotlib offset convention) — extracted
    # from the reference SVG so clustered labels fan out without overlapping.
    LABEL_OFFSETS = {
        "Surabaya": (5.1, -7.2), "Makassar": (5.1, -4.6),
        "Morotai":  (5.9,  1.3), "Galela":   (5.9, -2.1),
        "Maba":     (5.9, -5.4), "Weda":     (5.9, -8.8),
        "Kupang":   (3.7,  4.5), "Rote":     (3.7, -16.7),
        "Sabu":    (-22.5,  4.5),
    }

    def place_label(ax, name, lon, lat):
        dx, dy = LABEL_OFFSETS[name]
        ax.annotate(name, xy=(lon, lat), xytext=(dx, dy),
                    textcoords="offset points", ha="left", va="baseline",
                    fontsize=8, color=TEXT, zorder=6)

    for (name, lon, lat) in t9_ports:
        is_hub = name == "Surabaya"
        ax_a_map.scatter(lon, lat, s=28 if is_hub else 14,
                         c=MAROON if is_hub else PORT_GRAY,
                         edgecolors="white", lw=0.5, zorder=5)
        place_label(ax_a_map, name, lon, lat)

    for i in range(len(t9_ports) - 1):
        p1 = t9_ports[i]; p2 = t9_ports[i + 1]
        ax_a_map.annotate("", xy=(p2[1], p2[2]), xytext=(p1[1], p1[2]),
                          arrowprops=dict(arrowstyle="-|>", color=BLUE,
                                          lw=1.0, alpha=0.95, mutation_scale=12,
                                          capstyle="round", shrinkA=0, shrinkB=0),
                          zorder=3)
    p1 = t9_ports[-1]; p2 = t9_ports[0]
    arrow = FancyArrowPatch((p1[1], p1[2]), (p2[1], p2[2]),
                            connectionstyle="arc3,rad=-0.30",
                            arrowstyle="-|>", color=MAROON, mutation_scale=12,
                            lw=1.4, linestyle=(0, (4, 2)), alpha=0.95,
                            capstyle="round", zorder=4)
    ax_a_map.add_patch(arrow)

    ax_a_map.set_title("Case A — T-9 (Surabaya–Maluku Utara)",
                       fontsize=10, color=TEXT)
    ax_a_map.set_xlabel("Longitude (°E)", color=TEXT)
    ax_a_map.set_ylabel("Latitude (°)", color=TEXT)
    ax_a_map.set_xlim(110, 134)
    ax_a_map.set_ylim(-10, 5.5)
    ax_a_map.xaxis.set_major_locator(plt.MultipleLocator(5))
    ax_a_map.yaxis.set_major_locator(plt.MultipleLocator(2))
    _style_axes(ax_a_map)

    colors_a = [BLUE] * 5 + [MAROON]
    bars_a = ax_a_bar.bar(range(len(t9_leg_distances)), t9_leg_distances,
                          color=colors_a, edgecolor="white", lw=0.6)
    for i, (b, d) in enumerate(zip(bars_a, t9_leg_distances)):
        ax_a_bar.text(b.get_x() + b.get_width() / 2, b.get_height() + 30,
                      f"{d}", ha="center", fontsize=8, color=TEXT)
    ax_a_bar.set_xticks(range(len(t9_leg_labels)))
    ax_a_bar.set_xticklabels(t9_leg_labels, rotation=35, ha="right")
    ax_a_bar.set_ylabel("Leg distance (nm)", color=TEXT)
    ax_a_bar.set_ylim(0, 1500)
    ax_a_bar.yaxis.set_major_locator(plt.MultipleLocator(250))
    ax_a_bar.set_title("Return leg dominates: 1,320 nm vs ≤ 870 nm",
                       fontsize=10, color=TEXT)
    _style_axes(ax_a_bar)

    # ── CASE B: S-5A route (symmetric) ──────────────────────────
    s5a_ports = [
        ("Kupang",  123.58, -10.17),
        ("Rote",    122.98, -10.77),
        ("Sabu",    121.83, -10.50),
    ]
    s5a_leg_distances = [55, 88, 105]
    s5a_leg_labels = ["KPG→RTE", "RTE→SAB", "SAB→KPG"]

    for (name, lon, lat) in s5a_ports:
        is_hub = name == "Kupang"
        ax_b_map.scatter(lon, lat, s=28 if is_hub else 14,
                         c=MAROON if is_hub else PORT_GRAY,
                         edgecolors="white", lw=0.5, zorder=5)
        place_label(ax_b_map, name, lon, lat)

    full_loop = s5a_ports + [s5a_ports[0]]
    for i in range(len(full_loop) - 1):
        p1 = full_loop[i]; p2 = full_loop[i + 1]
        ax_b_map.annotate("", xy=(p2[1], p2[2]), xytext=(p1[1], p1[2]),
                          arrowprops=dict(arrowstyle="-|>", color=BLUE,
                                          lw=1.1, alpha=0.95, mutation_scale=12,
                                          capstyle="round", shrinkA=0, shrinkB=0),
                          zorder=3)

    ax_b_map.set_title("Case B — S-5A (Kupang–Rote–Sabu)",
                       fontsize=10, color=TEXT)
    ax_b_map.set_xlabel("Longitude (°E)", color=TEXT)
    ax_b_map.set_ylabel("Latitude (°)", color=TEXT)
    ax_b_map.set_xlim(121.0, 124.3)
    ax_b_map.set_ylim(-11.3, -9.7)
    ax_b_map.xaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax_b_map.yaxis.set_major_locator(plt.MultipleLocator(0.25))
    _style_axes(ax_b_map)

    colors_b = [BLUE] * 3
    bars_b = ax_b_bar.bar(range(len(s5a_leg_distances)), s5a_leg_distances,
                          color=colors_b, edgecolor="white", lw=0.6)
    for b, d in zip(bars_b, s5a_leg_distances):
        ax_b_bar.text(b.get_x() + b.get_width() / 2, b.get_height() + 4,
                      f"{d}", ha="center", fontsize=8, color=TEXT)
    ax_b_bar.set_xticks(range(len(s5a_leg_labels)))
    ax_b_bar.set_xticklabels(s5a_leg_labels, rotation=0)
    ax_b_bar.set_ylabel("Leg distance (nm)", color=TEXT)
    ax_b_bar.set_ylim(0, 160)
    ax_b_bar.yaxis.set_major_locator(plt.MultipleLocator(40))
    ax_b_bar.set_title("All legs comparable: 55-105 nm",
                       fontsize=10, color=TEXT)
    _style_axes(ax_b_bar)

    # Shared legend at top
    legend_items = [
        Line2D([0], [0], color=BLUE, lw=1.4, label="Outbound / symmetric legs"),
        Line2D([0], [0], color=MAROON, lw=1.4, linestyle=(0, (4, 2)),
               label="Return leg to hub"),
    ]
    leg = fig.legend(handles=legend_items, loc="upper center", ncol=2,
                     bbox_to_anchor=(0.5, 1.0), fontsize=9, frameon=False)
    for txt in leg.get_texts():
        txt.set_color(TEXT)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


if __name__ == "__main__":
    fig = figure_2_route_asymmetry()
    fig.savefig(os.path.join(save_folder, "Figure_2_Route_Asymmetry.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(save_folder, "Figure_2_Route_Asymmetry.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(save_folder, "Figure_2_Route_Asymmetry.svg"), bbox_inches="tight")
    print("✓ Figure 2 saved: Figure_2_Route_Asymmetry.png / .pdf / .svg")



✓ Figure 2 saved: Figure_2_Route_Asymmetry.png / .pdf / .svg


In [ ]:
# figure 4
routes = np.array(["S-4A","S-3","T-10","S-5","T-11","T-2","H-1","S-1","T-5","H-3",
                   "T-9","H-4","S-4B","H-2","T-12","T-20"])
dmax_nm = np.array([160,358,415,487,507,554,562,616,617,647,
                    710,715,745,766,818,1152], dtype=float)
dwt_cargo = np.array([1788.4,1378.1,1260.0,1110.7,1069.3,971.7,955.2,843.3,841.2,779.1,
                      648.5,638.2,576.1,532.4,424.6,-267.5], dtype=float)
feasible = dwt_cargo >= 1000

plt.rcParams.update({
    "font.family":"sans-serif","font.sans-serif":["DejaVu Sans","Arial","Helvetica"],
    "font.size":9.5,"axes.labelsize":11,"axes.titlesize":14,"axes.titleweight":"normal",
    "legend.fontsize":10,"axes.linewidth":0.7,"xtick.labelsize":9.5,"ytick.labelsize":9.5,
})
COL_CURVE="#2C4A66"; COL_VIABLE="#3A5F7D"; COL_INFEAS="#8B3A3A"
COL_REF="#8B7355"; COL_GRID="#E6E6E6"; COL_LEAD="#AAAAAA"

fig, ax = plt.subplots(figsize=(9.8,5.8))

coef = np.polyfit(dmax_nm, dwt_cargo, 1)
x_line = np.linspace(0,1200,300); y_line = np.polyval(coef, x_line)

ax.axhspan(1000,2050,color=COL_VIABLE,alpha=0.08,zorder=0)
ax.axhspan(-300,1000,color=COL_INFEAS,alpha=0.07,zorder=0)
ax.plot(x_line,y_line,color=COL_CURVE,lw=2.0,zorder=2)

ax.scatter(dmax_nm[feasible],dwt_cargo[feasible],s=58,color=COL_VIABLE,
           edgecolor="white",linewidth=0.8,zorder=4)
ax.scatter(dmax_nm[~feasible],dwt_cargo[~feasible],s=64,marker="x",
           color=COL_INFEAS,linewidth=2.0,zorder=5)

ax.axhline(1000,color=COL_REF,lw=1.0,linestyle=(0,(4,3)),zorder=1)
ax.axvline(542,color=COL_REF,lw=1.0,linestyle=(0,(4,3)),zorder=1)
ax.text(1130,1038,"1,000 t cargo target",ha="right",va="bottom",
        color=COL_REF,fontsize=9.5,style="italic")
ax.text(535,1990,"542 nm threshold",ha="right",va="top",
        color=COL_REF,fontsize=9.5,style="italic")

# ---- Viable labels (kept close, they were already clean) ----
viable_offsets = {"S-4A":(-12,12),"S-3":(-10,14),"T-10":(-8,14),"S-5":(-12,14),"T-11":(-4,16)}
for r,x,y,ok in zip(routes,dmax_nm,dwt_cargo,feasible):
    if ok:
        dx,dy = viable_offsets[r]
        ax.annotate(r,xy=(x,y),xytext=(dx,dy),textcoords="offset points",
                    ha="left" if dx>=0 else "right", va="bottom" if dy>=0 else "top",
                    color=COL_VIABLE,fontsize=9.2,zorder=6)

# ---- Infeasible labels: shelf in the lower-left triangle ----
infeas_idx = [i for i in range(len(routes)) if not feasible[i]]
cluster = [i for i in infeas_idx if dmax_nm[i] < 1000]      # 10 packed routes
cluster.sort(key=lambda i: dmax_nm[i])                       # order along the curve

# Shelf: evenly spaced x anchors on a line PARALLEL to the curve (constant
# vertical clearance C below it) so every label sits the same distance
# below-left of its marker and the leaders stay short and uniform.
x_start, x_end = 460, 820
C = 260.0                      # vertical clearance below the curve (tonnes)
n = len(cluster)
shelf_x = np.linspace(x_start, x_end, n)
shelf_y = coef[0] * shelf_x + coef[1] - C

for k, i in enumerate(cluster):
    px, py = dmax_nm[i], dwt_cargo[i]
    lx, ly = shelf_x[k], shelf_y[k]
    ax.plot([px, lx], [py, ly], color=COL_LEAD, lw=0.7, zorder=3)
    ax.text(lx, ly, routes[i], ha="center", va="center",
            color=COL_INFEAS, fontsize=9.2, zorder=6)

# ---- T-20: direct label, isolated (placed clear of the curve, below-left) ----
i20 = int(np.where(routes=="T-20")[0][0])
ax.plot([dmax_nm[i20], 1010],[dwt_cargo[i20], -185], color=COL_LEAD, lw=0.7, zorder=3)
ax.text(1010, -185, "T-20\n(actual: \u2212268 t)", ha="center", va="center",
        color=COL_INFEAS, fontsize=9.2, zorder=6)

ax.set_title("Payload-distance trade-off under baseline assumptions", pad=14)
ax.set_xlabel(r"Maximum continuous leg distance, $D_{max}$ (nautical miles)")
ax.set_ylabel("Residual cargo deadweight (tonnes)")
ax.set_xlim(0,1200); ax.set_ylim(-300,2000)
ax.set_xticks(np.arange(0,1201,100))
ax.set_yticks([-300,-50,200,450,700,950,1200,1450,1700,1950])
ax.grid(True,axis="y",color=COL_GRID,lw=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.tick_params(colors="#666666")

legend_handles=[
    Line2D([0],[0],color=COL_CURVE,lw=2.0,label="Model trade-off curve"),
    Line2D([0],[0],marker="o",color="none",markerfacecolor=COL_VIABLE,
           markeredgecolor="white",markeredgewidth=0.8,markersize=8,label="Viable route (5)"),
    Line2D([0],[0],marker="x",color=COL_INFEAS,lw=0,markersize=8,markeredgewidth=2.0,
           label="Infeasible route (11)"),
]
ax.legend(handles=legend_handles,loc="upper right",frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(save_folder, "Figure_4_Payload_Distance_Tradeoff.png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(save_folder, "Figure_4_Payload_Distance_Tradeoff.svg"), bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved")


saved


In [ ]:
# figure 5
"""
Figure 5 (Scatter Plot v2): Improved with all route labels, legend box, clarified ceiling label

Changes:
1. Legend moved to right side with box frame
2. All 16 routes labeled (viable with arrows, infeasible with leader lines where clustered)
3. 542 nm ceiling label clarified: "542 nm: max distance for ≥1,000 t cargo"
"""

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":         8.5,
    "axes.labelsize":    9.5,
    "axes.titlesize":    11,
    "axes.titleweight":  "normal",
    "axes.linewidth":    0.6,
    "xtick.labelsize":   8.5,
    "ytick.labelsize":   8.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size":  3.0,
    "ytick.major.size":  3.0,
    "legend.fontsize":   8.5,
    "legend.frameon":    True,
    "legend.fancybox":   False,
    "legend.edgecolor":  "#CCCCCC",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         False,
    "axes.axisbelow":    True,
    "lines.linewidth":   1.0,
})

# Color palette
COL_VIABLE  = "#2E8B57"    # sea green
COL_INFEAS  = "#C0392B"    # clear brick red
COL_REF     = "#8A8A8A"    # neutral grey
COL_TEXT    = "#1A1A1A"
COL_AXIS    = "#555555"
COL_GRID    = "#E8E8E8"


def detect_clusters(distances, total_dw, threshold_x=50, threshold_y=30):
    """
    Identify routes that are clustered (close in distance and/or deadweight).
    Returns a list of cluster groups, each containing indices of routes in that cluster.
    """
    n = len(distances)
    clustered = [False] * n
    clusters = []
    
    for i in range(n):
        if clustered[i]:
            continue
        # Start a new cluster with route i
        group = [i]
        for j in range(i + 1, n):
            if clustered[j]:
                continue
            dx = abs(distances[j] - distances[i])
            dy = abs(total_dw[j] - total_dw[i])
            if dx < threshold_x or dy < threshold_y:
                group.append(j)
                clustered[j] = True
        
        if len(group) > 1:
            clusters.append(group)
            clustered[i] = True
    
    return clusters


def build_figure_5_scatter_v2():
    """
    Scatter plot v2: all routes labeled, legend in box on right, clarified ceiling label.
    """
    fig, ax = plt.subplots(figsize=(8.5, 5.4))

    routes_data = {
        'code': ['S-4A', 'S-3', 'T-10', 'S-5', 'T-11',
                 'T-2', 'H-1', 'S-1', 'T-5', 'H-3',
                 'T-9', 'H-4', 'S-4B', 'H-2', 'T-12', 'T-20'],
        'distance': [160, 358, 415, 487, 507,
                     554, 562, 616, 617, 647,
                     710, 715, 745, 766, 818, 1152],
        'cargo': [1789, 1380, 1262, 1113, 1072,
                  972, 955, 843, 841, 779,
                  649, 638, 576, 532, 425, -268],
        'battery_mass': [148, 257, 319, 383, 414,
                         466, 483, 580, 582, 641,
                         725, 736, 801, 856, 974, 1355],
    }

    distances = np.array(routes_data['distance'], dtype=float)
    cargo = np.array(routes_data['cargo'], dtype=float)
    battery = np.array(routes_data['battery_mass'], dtype=float)
    total_dw = cargo + battery

    # ---- Plot points ----
    viable_mask = np.array([i < 5 for i in range(len(distances))])
    
    # Viable routes (green triangles)
    ax.scatter(distances[viable_mask], total_dw[viable_mask],
               color=COL_VIABLE, marker='^', s=130, alpha=0.85,
               edgecolors='white', linewidth=1.2, zorder=5,
               label='Viable route (cargo ≥ 1,000 t)')
    
    # Infeasible routes (red circles)
    ax.scatter(distances[~viable_mask], total_dw[~viable_mask],
               color=COL_INFEAS, marker='o', s=110, alpha=0.75,
               edgecolors='white', linewidth=1.0, zorder=4,
               label='Infeasible route (cargo < 1,000 t)')

    # ---- Reference lines ----
    # 2,000 t DWT hull limit
    ax.axhline(2000, color=COL_REF, lw=0.9, linestyle='-', alpha=0.7, zorder=1)
    ax.text(20, 2040, '2,000 t hull DWT limit', fontsize=8.5, color=COL_REF,
            ha='left', va='bottom', weight='normal')

    # 542 nm ceiling — clarified label
    ax.axvline(542, color=COL_REF, lw=0.9, linestyle='--', alpha=0.7, zorder=2)
    ax.text(548, 1950, '542 nm: max distance\nfor ≥1,000 t cargo', fontsize=8, color=COL_REF,
            ha='left', va='center', rotation=90)

    # ---- Label all routes ----
    
    # Detect clusters in infeasible routes (indices 5–15)
    infeasible_dist = distances[5:]
    infeasible_dw = total_dw[5:]
    clusters = detect_clusters(infeasible_dist, infeasible_dw, threshold_x=50, threshold_y=30)
    
    # Mark which infeasible routes are in clusters
    clustered_infeasible = set()
    for cluster_group in clusters:
        for idx in cluster_group:
            clustered_infeasible.add(idx + 5)  # Adjust for 0-indexing
    
    # Label viable routes (direct with arrows, except S-5)
    for i in range(5):
        dist = distances[i]
        dw = total_dw[i]
        code = routes_data['code'][i]
        
        # Special case: S-5 (index 3) → move to bottom-left to avoid 542 nm line overlap
        if code == 'S-5':
            ax.text(dist - 35, dw - 50, code, fontsize=7.5, color=COL_TEXT,
                   ha='right', va='top', weight='normal', zorder=8,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                            edgecolor='none', alpha=0.7))
        else:
            # Standard: label offset above/right with arrow
            ax.annotate(
                code,
                xy=(dist, dw), xycoords='data',
                xytext=(dist + 20, dw + 40), textcoords='data',
                fontsize=7.5, color=COL_TEXT, weight='normal',
                ha='left', va='bottom',
                arrowprops=dict(arrowstyle='->', color=COL_VIABLE, lw=0.5, alpha=0.6,
                              shrinkA=3, shrinkB=2),
                zorder=8,
            )
    
    # Label infeasible routes (with or without leader lines)
    for i in range(5, 16):
        dist = distances[i]
        dw = total_dw[i]
        code = routes_data['code'][i]
        
        # Special case: T-5 (index 8) → move to bottom-left to avoid 542 nm line overlap
        if code == 'T-5':
            ax.text(dist - 35, dw - 50, code, fontsize=7, color=COL_TEXT,
                   ha='right', va='top', zorder=7,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                            edgecolor='none', alpha=0.7))
        elif i in clustered_infeasible:
            # Use leader line for other clustered routes
            # Determine label position based on which cluster and position within cluster
            # For simplicity: alternate above/below or left/right
            
            if code in ['T-2', 'H-1']:  # Very close, same y
                # T-2 on left, H-1 on right, both slightly above
                if code == 'T-2':
                    label_x, label_y = dist - 20, dw + 50
                else:
                    label_x, label_y = dist + 20, dw + 50
            elif code in ['H-3']:  # Part of T-5 cluster but keep above
                label_x, label_y = dist + 30, dw - 60
            elif code in ['T-9', 'H-4']:  # Very close, same y
                if code == 'T-9':
                    label_x, label_y = dist - 25, dw + 50
                else:
                    label_x, label_y = dist + 25, dw + 50
            elif code in ['S-4B', 'H-2']:  # Close, slightly separated
                if code == 'S-4B':
                    label_x, label_y = dist - 30, dw - 60
                else:
                    label_x, label_y = dist + 30, dw + 60
            else:
                # Fallback for other routes
                label_x, label_y = dist + 25, dw + 50
            
            # Draw leader line
            ax.plot([dist, label_x], [dw, label_y],
                   color=COL_INFEAS, lw=0.4, alpha=0.5, zorder=6)
            
            # Label text
            ax.text(label_x, label_y, code, fontsize=7, color=COL_TEXT,
                   ha='center', va='center', bbox=dict(boxstyle='round,pad=0.3',
                                                        facecolor='white', edgecolor='none',
                                                        alpha=0.7),
                   zorder=7)
        else:
            # Isolated infeasible routes — direct label
            ax.text(dist + 15, dw + 30, code, fontsize=7, color=COL_TEXT,
                   ha='left', va='bottom', alpha=0.8, zorder=7)

    # ---- Axes ----
    ax.set_xlabel('Maximum continuous leg distance D$_{max}$ (nautical miles)',
                  fontsize=9.5, color=COL_TEXT, labelpad=8)
    ax.set_ylabel('Total deadweight (cargo + battery) (tonnes)',
                  fontsize=9.5, color=COL_TEXT, labelpad=8)

    ax.set_xlim(-20, 1200)
    ax.set_ylim(800, 2200)

    ax.xaxis.set_major_locator(plt.MultipleLocator(200))
    ax.yaxis.set_major_locator(plt.MultipleLocator(200))
    ax.tick_params(colors=COL_AXIS, labelsize=8.5)

    # Light grid
    ax.grid(True, color=COL_GRID, linewidth=0.5, linestyle='-', alpha=0.6,
            axis='y', zorder=0)

    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(COL_AXIS)
        ax.spines[spine].set_linewidth(0.6)

    # ---- Legend — moved to right side with box, positioned above 2000 t line ----
    legend = ax.legend(loc='center right', bbox_to_anchor=(0.98, 0.93),
                       fontsize=8.5, handlelength=1.8, labelspacing=1.2,
                       frameon=True, fancybox=False, edgecolor='#CCCCCC',
                       facecolor='white', framealpha=0.95)
    legend.set_zorder(100)

    ax.set_title('Route viability: total deadweight vs. distance',
                 fontsize=11, color=COL_TEXT, pad=12, weight='normal')

    fig.subplots_adjust(left=0.10, right=0.97, top=0.91, bottom=0.12)

    png_path = os.path.join(save_folder, "Figure_5_Battery_Mass_Cascade_SCATTER.png")
    svg_path = os.path.join(save_folder, "Figure_5_Battery_Mass_Cascade_SCATTER.svg")
    fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white", edgecolor="none")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close(fig)
    
    print(f"✓ Saved {png_path}")
    print(f"✓ Saved {svg_path}")
    return png_path, svg_path


if __name__ == "__main__":
    print("=" * 80)
    print("FIGURE 5 SCATTER PLOT v2 — All routes labeled, legend in box, clarified ceiling")
    print("=" * 80)
    print("\nImprovements:")
    print("  1. Legend moved to right side with box frame")
    print("  2. All 16 routes labeled (viable with arrows, infeasible with smart placement)")
    print("  3. Clustered routes use leader lines to avoid overlap")
    print("  4. 542 nm label clarified: 'max distance for ≥1,000 t cargo'")
    print()
    
    build_figure_5_scatter_v2()
    
    print("\n" + "=" * 80)
    print("Scatter plot v2 complete. Ready for journal submission.")
    print("=" * 80)


FIGURE 5 SCATTER PLOT v2 — All routes labeled, legend in box, clarified ceiling

Improvements:
  1. Legend moved to right side with box frame
  2. All 16 routes labeled (viable with arrows, infeasible with smart placement)
  3. Clustered routes use leader lines to avoid overlap
  4. 542 nm label clarified: 'max distance for ≥1,000 t cargo'

✓ Saved output_data\Figure_5_Battery_Mass_Cascade_SCATTER.png
✓ Saved output_data\Figure_5_Battery_Mass_Cascade_SCATTER.svg

Scatter plot v2 complete. Ready for journal submission.


In [ ]:
# figure 6

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 12, "legend.fontsize": 8.5,
})
COL_BASELINE = "#244A73"
COL_ADJUSTED = "#8B5A2B"

route_dmax_nm = np.array([160,358,415,487,507,554,562,616,617,647,
                          710,715,745,766,818,1152])
cargo_data = np.array([1789,1380,1262,1113,1072,972,955,843,841,779,
                       649,638,576,532,425,-268])
A = np.vstack([np.ones_like(route_dmax_nm), -route_dmax_nm]).T
P0, slope = np.linalg.lstsq(A, cargo_data, rcond=None)[0]
RHO0 = 0.14
def D_max(rho, cargo): return (P0 - cargo) * rho / (RHO0 * slope)
def count_viable(rho, cargo): return int(np.sum(route_dmax_nm <= D_max(rho, cargo)))

baseline = (0.14, 1000); adjusted = (0.16, 840)
n_base = count_viable(*baseline); n_adj = count_viable(*adjusted)

rho_cells   = np.round(np.arange(0.10, 0.181, 0.01), 2)
cargo_cells = np.arange(700, 1201, 50)
grid = np.array([[count_viable(r, c) for r in rho_cells] for c in cargo_cells])

colors = ["#d95f5f","#f28e63","#f8c87c","#f4e6a1","#d9e9a5","#9fcd8b","#4f9d69"]
cmap = LinearSegmentedColormap.from_list("muted_viability", colors, N=256)
norm = BoundaryNorm(np.arange(0, 17, 1), cmap.N, clip=True)

fig, ax = plt.subplots(figsize=(8.6, 5.6))
im = ax.imshow(grid, origin="lower", aspect="auto", cmap=cmap, norm=norm,
               extent=[rho_cells[0]-0.005, rho_cells[-1]+0.005,
                       cargo_cells[0]-25, cargo_cells[-1]+25])

for i, c in enumerate(cargo_cells):
    for j, r in enumerate(rho_cells):
        val = grid[i, j]
        ax.text(r, c, str(val), ha="center", va="center",
                fontsize=7.5, color=("#222222" if 3 <= val <= 13 else "white"))

# Scenario stars
ax.scatter([baseline[0]], [baseline[1]], marker="*", s=300, color=COL_BASELINE,
           edgecolors="white", linewidth=1.2, zorder=6)
ax.scatter([adjusted[0]], [adjusted[1]], marker="*", s=300, color=COL_ADJUSTED,
           edgecolors="white", linewidth=1.2, zorder=6)

# Annotations WITH white background boxes so they read cleanly over cells
ax.annotate(f"Baseline\n(0.14 MWh/t, 1000 t)\n{n_base} viable routes",
            xy=baseline, xytext=(0.102, 1135),
            fontsize=8.0, color=COL_BASELINE, ha="left", va="center", weight="bold",
            arrowprops=dict(arrowstyle="-|>", color=COL_BASELINE, lw=1.0, ls="--"),
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                      edgecolor=COL_BASELINE, alpha=0.92), zorder=8)
ax.annotate(f"Adjusted operating case\n(0.16 MWh/t, 840 t)\n{n_adj} viable routes",
            xy=adjusted, xytext=(0.101, 735),
            fontsize=8.0, color="#4A2A12", ha="left", va="center", weight="bold",
            arrowprops=dict(arrowstyle="-|>", color="#4A2A12", lw=1.0, ls="--"),
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                      edgecolor="#8B5A2B", alpha=0.92), zorder=8)

ax.set_title("Viable route count across battery density and cargo threshold")
ax.set_xlabel(r"Battery gravimetric energy density, $\rho_{grav}$ (MWh/tonne)")
ax.set_ylabel(r"Cargo viability threshold, $DWT_{target}$ (tonnes)")
ax.set_xticks(rho_cells); ax.set_yticks(cargo_cells)
cbar = fig.colorbar(im, ax=ax, pad=0.02, ticks=np.arange(0, 17, 2))
cbar.set_label("Viable routes")
fig.tight_layout()
fig.savefig(os.path.join(save_folder, "Figure_6_Sensitivity_Heatmap.png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(save_folder, "Figure_6_Sensitivity_Heatmap.svg"), bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Baseline -> {n_base} routes ; Adjusted -> {n_adj} routes")
print("Saved canonical Figure 6 heatmap")


Baseline -> 5 routes ; Adjusted -> 10 routes
Saved canonical Figure 6 heatmap


In [ ]:
# figure 7
df = pd.read_csv("Tol_Laut_Sizing_Engine_Results.csv")
order = ['S-4A','S-3','T-10','S-5','T-11','T-2','H-1','S-1','T-5','H-3',
         'T-9','H-4','S-4B','H-2','T-12','T-20']
df = df.set_index('Route_Code').loc[order].reset_index()

T_DWELL=4.0; ETA_CHARGER=0.92; DOD=0.80
codes  = df['Route_Code'].tolist()
p_fast = (df['E_installed_MWh']*DOD*1000/(T_DWELL*ETA_CHARGER)/1000).tolist()
p_swap = (df['P_port_kW']/1000.0).tolist()

fig, ax = plt.subplots(figsize=(11,6))
x = np.arange(len(codes)); width = 0.40

ax.bar(x - width/2, p_fast, width, color="#c0392b", alpha=0.85,
       label="Direct fast-charge (4 h dwell)", edgecolor="white")
ax.bar(x + width/2, p_swap, width, color="#27ae60", alpha=0.85,
       label="Container swap (full arrival interval)", edgecolor="white")

# Reference lines
ax.axhline(5, color="#444", lw=1.5, linestyle="--", alpha=0.9)
ax.text(15.7, 5.8, "Typical 3T grid capacity (~5 MW)",
        ha="right", fontsize=9, color="#444", fontweight="bold")
ax.axhline(10, color="#888", lw=1.0, linestyle=":", alpha=0.8)
# (2) readable hub-capacity annotation: darker, larger, white backing box
ax.text(15.7, 11.0, "Larger regional hub capacity (~10 MW)",
        ha="right", fontsize=9, color="#444", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                  edgecolor="none", alpha=0.75))

# Fast-charge labels (the dramatic ones, unchanged behaviour)
for i, p in enumerate(p_fast):
    if p > 25:
        ax.text(i - width/2, min(p + 1.5, 78), f"{p:.0f}",
                ha="center", fontsize=8, color="#c0392b", fontweight="bold")

# (1) Container-swap value labels on top of each green bar
for i, p in enumerate(p_swap):
    ax.text(i + width/2, p + 1.0, f"{p:.2f}", ha="center", va="bottom",
            fontsize=6.6, color="#1e8449", rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(codes, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Continuous shore power demand (MW)", fontsize=10.5)
ax.set_xlabel("Tol Laut route", fontsize=10.5)
# (3) new title
ax.set_title("Comparison of shore-side charging demand under direct fast-charging "
             "and containerized battery-swapping assumptions",
             fontsize=11, color="#1F3864", pad=10)
ax.set_ylim(0, 85)
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.25, linestyle=":", axis="y")
plt.tight_layout()
fig.savefig(os.path.join(save_folder, "Figure_7_Charging_Power_Comparison.png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(save_folder, "Figure_7_Charging_Power_Comparison.svg"), bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved")

saved


In [21]:
# figure 8

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans","Arial"],
    "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10.5,
    "legend.fontsize": 8, "axes.linewidth": 0.8,
})

# Clean, bright, accessible palette (sky blue / amber / coral / green on white)
C_DEMAND   = "#5B9BD5"; C_DEMAND_L = "#2F6DA4"   # residential demand (sky blue)
C_UTIL     = "#F4B942"; C_UTIL_L   = "#D99B1E"   # utilized solar (amber-gold)
C_CURTAIL  = "#E8736A"; C_CURT_L   = "#CF5249"   # curtailed solar (coral red)
C_CHARGE   = "#5BA053"; C_CHARGE_L = "#3F8038"   # ship-charging load (green)
C_SOLARLN  = "#ED9121"                            # solar generation line (orange)
C_TEXT     = "#222222"; C_GRID = "#E2E2E2"; C_AXIS = "#555555"

h = np.linspace(0, 24, 400)
demand = (200 + 130*np.exp(-((h-7.5)/1.6)**2)
          + 1200*np.exp(-((h-20)/2.0)**2) + 60*np.exp(-((h-12)/4.5)**2))
solar = 1800*np.exp(-((h-12)/3.0)**2); solar[(h < 6) | (h > 18)] = 0.0
utilized = np.minimum(solar, demand)
charge_top = np.where((h >= 8) & (h <= 17), 1450.0, np.nan)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.4, 6.4), sharex=True)

# ---------- Top: without scheduled charging load ----------
ax1.fill_between(h, 0, demand, color=C_DEMAND, alpha=0.70, lw=0)
ax1.fill_between(h, 0, utilized, color=C_UTIL, alpha=0.85, lw=0)
ax1.fill_between(h, utilized, solar, color=C_CURTAIL, alpha=0.80, lw=0)
ax1.plot(h, solar, color=C_CURT_L, lw=1.3)
ax1.plot(h, demand, color=C_DEMAND_L, lw=1.4)
ax1.set_title("Without scheduled charging load: daytime solar generation is curtailed",
              color=C_TEXT, pad=8)
ax1.set_ylabel("Power (kW)", color=C_TEXT)
leg1 = [Patch(facecolor=C_DEMAND, alpha=0.70, edgecolor=C_DEMAND_L, label="3T residential demand"),
        Patch(facecolor=C_UTIL, alpha=0.85, edgecolor=C_UTIL_L, label="Utilized solar generation"),
        Patch(facecolor=C_CURTAIL, alpha=0.80, edgecolor=C_CURT_L, label="Curtailed solar generation")]
ax1.legend(handles=leg1, loc="upper left", frameon=True, framealpha=0.95, edgecolor="#CCCCCC")

# ---------- Bottom: with scheduled charging load ----------
ax2.fill_between(h, 0, demand, color=C_DEMAND, alpha=0.70, lw=0)
ax2.fill_between(h, demand, charge_top, where=(h >= 8) & (h <= 17),
                 color=C_CHARGE, alpha=0.72, lw=0)
# crisp outline around the charging block
mask = (h >= 8) & (h <= 17)
ax2.plot(h[mask], charge_top[mask], color=C_CHARGE_L, lw=1.4)
ax2.plot([8, 8], [demand[np.argmin(abs(h-8))], 1450], color=C_CHARGE_L, lw=1.4)
ax2.plot([17, 17], [demand[np.argmin(abs(h-17))], 1450], color=C_CHARGE_L, lw=1.4)
ax2.plot(h, demand, color=C_DEMAND_L, lw=1.4)
ax2.plot(h, solar, color=C_SOLARLN, lw=2.0, linestyle="--")
ax2.set_title("With scheduled ship-charging load: daytime solar generation is fully utilized",
              color=C_TEXT, pad=8)
ax2.set_ylabel("Power (kW)", color=C_TEXT); ax2.set_xlabel("Hour of day", color=C_TEXT)
leg2 = [Patch(facecolor=C_DEMAND, alpha=0.70, edgecolor=C_DEMAND_L, label="3T residential demand"),
        Patch(facecolor=C_CHARGE, alpha=0.72, edgecolor=C_CHARGE_L, label="Scheduled ship-charging load"),
        Line2D([0],[0], color=C_SOLARLN, lw=2.0, ls="--", label="Solar generation (utilized)")]
ax2.legend(handles=leg2, loc="upper left", frameon=True, framealpha=0.95, edgecolor="#CCCCCC")

for ax in (ax1, ax2):
    ax.set_xlim(0, 24); ax.set_ylim(0, 2100)
    ax.set_xticks(np.arange(0, 25, 3)); ax.set_yticks(np.arange(0, 2101, 500))
    ax.grid(True, axis="y", color=C_GRID, lw=0.6)
    ax.tick_params(colors=C_AXIS)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(C_AXIS)

fig.suptitle("Scheduled ship-charging load and daytime solar utilization in 3T microgrids",
             fontsize=12, color=C_TEXT, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(os.path.join(save_folder, "Figure_8_Microgrid_Solar_Utilization.png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(save_folder, "Figure_8_Microgrid_Solar_Utilization.svg"), bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved")


saved
